In [15]:
from ingest import load_faq_data
documents = load_faq_data()

In [16]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

83

In [17]:
documents = documents_llm

In [18]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


The ID becomes the label in our ground truth dataset. 

This is why every record needs a stable ID. If you can't uniquely identify a document, you can't tell whether search retrieved the right one. When you build your own evaluation set, assign an ID to each record in your knowledge base first.

In [19]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [20]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [21]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [22]:
import json

user_prompt = json.dumps(doc)

In [23]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called responses.create and read response.output_text. For structured output we switch to responses.parse and pass text_format=Questions, which tells the API to return our class instead of free text.

### Below is an updated version of the course, using groq instad

In [24]:
response = openai_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "Questions_schema", # Changed "Questions schema" to "Questions_schema"
            "schema": Questions.model_json_schema()
        }
    }
)

In [25]:
raw_result = json.loads(response.choices[0].message.content or "{}")
result = Questions.model_validate(raw_result)
print(result.model_dump_json(indent=2))

{
  "questions": [
    "Q: Can I enroll in the LLM Zoomcamp now even if I missed the start, and will I still get a certificate? A: Yes, you can still join, but you must submit your project before the submission deadline to earn a certificate.",
    "Q: Is there a cutoff date for getting the certificate after I finish the course? A: You need to submit your final project while we’re still accepting submissions; otherwise you won’t receive the certificate.",
    "Q: Do I need to be a beginner to start the LLM Zoomcamp, or can I join mid‑way? A: The course is open‑ended; you can start at any time, just make sure to complete the project before the deadline for certification.",
    "Q: What happens if I join now but miss the project deadline—can I still get access to the materials? A: You’ll still have access to all lectures and resources, but the certificate is only awarded to those who submit a project before the deadline.",
    "Q: Are there any restrictions on late project submissions fo

In [26]:
from groq_evaluation_utils import llm_structured

In [27]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I enroll in the LLM ZoomCamp now even though it already started?', 'Is it still possible to get a certificate if I join the course late?', 'Do I need to submit my project to earn the certificate after enrolling late?', 'What’s the deadline for project submissions for a certificate?', 'If I start the course now, can I still earn the credential?']


# Note: llama-3.3-70b-versatile does not handle json format outputs, so use openai/gpt-oss-120b

In [28]:
usage

CompletionUsage(completion_tokens=349, prompt_tokens=332, total_tokens=681, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=256, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=256), queue_time=0.194111496, prompt_time=0.173087028, completion_time=0.765543729, total_time=0.938630757)

In [29]:
# Assuming 'usage' is your CompletionUsage object
print(f"Total Tokens: {usage.total_tokens}")
print(f"Completion Tokens: {usage.completion_tokens}")
print(f"Prompt Tokens: {usage.prompt_tokens}")

Total Tokens: 681
Completion Tokens: 349
Prompt Tokens: 332


In [30]:
# or:
# Convert the entire object to a dictionary
usage_dict = usage.model_dump()

# Access values like a dictionary
print(usage_dict["total_tokens"])

681


In [31]:
from groq_evaluation_utils import calc_price

In [32]:
# reload groq_evaluation_utils to get the updated calc_price function
import importlib
import groq_evaluation_utils
importlib.reload(groq_evaluation_utils)

<module 'groq_evaluation_utils' from '/Users/nhubert/Documents/llm-zoomcamp-code/04-orchestration/groq_evaluation_utils.py'>

In [33]:
from groq_evaluation_utils import calc_price

In [34]:
cost = calc_price(usage)

print(cost.total_cost)   # Works!
print(cost.input_cost)   # Works!

0.0018195
0.000249


In [35]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I enroll in the LLM ZoomCamp now even though it already started?',
  'document': '74eb249bbf'},
 {'question': 'Is it still possible to get a certificate if I join the course late?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project to earn the certificate after enrolling late?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for project submissions for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course now, can I still earn the credential?',
  'document': '74eb249bbf'}]

In [36]:
from groq_evaluation_utils import llm_structured_retry
importlib.reload(groq_evaluation_utils)

<module 'groq_evaluation_utils' from '/Users/nhubert/Documents/llm-zoomcamp-code/04-orchestration/groq_evaluation_utils.py'>

In [37]:
from groq_evaluation_utils import llm_structured_retry

In [38]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [39]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

This works, but it runs one LLM call after another. Running it for all documents this way would take too long. ==> **parallel processing**

## Parallel processing

Running the calls one after another wastes most of the time waiting on the network. Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. We process the documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

In [40]:
from concurrent.futures import ThreadPoolExecutor
from groq_evaluation_utils import map_progress

In [41]:
documents = documents[:9]  # Limit to 9 documents for testing, because I use Groq free tier

In [42]:
with ThreadPoolExecutor(max_workers=3) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/9 [00:00<?, ?it/s]

In [43]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

45

In [45]:
from groq_evaluation_utils import calc_price

# Simplified calculation using a generator expression
total_cost = sum(calc_price(usage).total_cost for usage in usages)

print(total_cost)

0.01629525


In [46]:
from groq_evaluation_utils import calc_total_price

calc_total_price(usages)

0.01629525

In [47]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [49]:
df_ground_truth.to_csv("ground_truth-new.csv", index=False)